In [352]:
import math
class myNode:
  def __init__(self, val,_children=(),_op = '',_backward = lambda: None):
    self.val = val
    self.grad = 0.0
    self._prev = set(_children)
    self._op = _op
    self._backward = _backward

  def __add__(self,other):
    other = other if isinstance(other,myNode) else myNode(other)
    out = myNode(self.val + other.val,(self,other),'+')
    def _backward():
      self.grad+= 1.0 * out.grad
      other.grad+= 1.0 * out.grad
    out._backward = _backward
    return out

  def __sub__(self,other):
    return self+(-other)

  def __rmul__(self, other):
    return self*other

  def __radd__(self,other):
    return self+other

  def __mul__(self,other):
    other = other if isinstance(other,myNode) else myNode(other)
    out = myNode(self.val * other.val,(self,other),'*')
    def _backward():
      self.grad += other.val * out.grad
      other.grad += self.val * out.grad
    out._backward = _backward
    return out
  def __neg__(self):
    return (self*(-1))

  def __pow__(self,other):
    out = myNode(self.val**other,(self,),f'**{other}')
    def _backward():
      self.grad += other * (self.val**(other-1)) * out.grad
    out._backward = _backward
    return out

  def __truediv__(self,other):
    return myNode(self*(other**-1))

  def tanh(self):
    x = self.val;
    i = math.exp(2*x)
    res = (i - 1)/(i+1)
    out = myNode(res,(self,),'tanh')
    def _backward():
      self.grad += (1 - res**2) * out.grad
    out._backward = _backward
    return out



  def build_backward(self):
    topo = []
    vis = set()
    def build_topo(self):
      if self not in vis:
        vis.add(self)
        for child in self._prev:
          build_topo(child)
        topo.append(self)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

  def __repr__(self):
    return f"myNode(val={self.val}, grad={self.grad})"



In [353]:
import graphviz as gr

G = gr.Digraph()
G.attr(rankdir='LR')

vis = set()
nodes = []
visitedNodes = []

def build_graph(root):
  if root not in vis:
    vis.add(root)
    G.node(str(id(root)), label=f"{{{str(id(root))}|val: {root.val} | grad: {root.grad} }}",shape = 'record')
    if(root._op):
      op_node_id = str(id(root)) + root._op
      G.node(op_node_id, label=f"{{{str(root._op)}}}",shape = 'record')
      G.edge(op_node_id,str(id(root)))
      for child in root._prev:
        G.edge(str(id(child)),op_node_id)
        build_graph(child)

    return G


In [354]:


a = myNode(2.4)
b = myNode(3.0)
c = a*b
# d = b * c
# d
# f = d + e
# g = f / i


# g.grad = 1.0
# g.build_backward()



# d.label = 'd'
# # d._show_()
# G = build_graph(g)
# G



In [355]:
import torch

In [356]:
a = torch.Tensor([2.4]).double() ; a.requires_grad = True
b = torch.Tensor([1.8]).double() ; b.requires_grad = True
c = torch.Tensor([3.0]).double() ; c.requires_grad = True
e = torch.Tensor([1.5]).double() ; e.requires_grad = True
i = torch.Tensor([1.7]).double() ; i.requires_grad = True

d = b*c + a
f = d+e
g = f/i
g.backward()
print(g.data.item())
print(c.grad.item())

5.470588053799011
1.0588234716633216


In [357]:
from IPython.core.formatters import IPythonDisplayFormatter
import random
class Neuron:
  def __init__(self,n):
    self.w = [myNode(random.uniform(-1,1)) for _ in range(n)]
    self._b = myNode(random.uniform(-1,1))
  def _parameters_(self):
    return self.w + [self._b]
  def __call__(self,x):
    inter = sum((wi*item for wi,item in zip(self.w,x)),self._b)
    return inter.tanh()

class Layer():
  def __init__(self,nin,nouts):
    self.neurons = [Neuron(nin) for _ in range(nouts)]
  def _params_(self):
    params = []
    for neuron in self.neurons:
      params.append(neuron._parameters_())
    return params
  def __call__(self,x):
    outs = [neuron(x) for neuron in self.neurons]
    return outs

class MLP():
  def __init__(self,nin,nouts):
   sizeArray = [nin] + nouts
   self.layers = [Layer(sizeArray[i],sizeArray[i+1]) for i in range(len(nouts))]
  def __call__(self,x):
    for layer in self.layers:
      x = layer(x)
    return x[0] if len(x) == 1 else x
  def _weights_(self):
    out = []
    for layer in self.layers:
      for pr in layer._params_():
       out.extend(pr)
    return out

# output.build_backward()

# G1 = build_graph(output)

# G1

# a = Layer(2,3)asdf
# a(x)





In [561]:

x = [2.0,1.5,-0.5]
a = MLP(3,[4,4,1])
a(x)
# output = a(x)
# output.build_backward()
# build_graph(output)
# a._weights_()

myNode(val=0.5620860424108158, grad=0.0)

In [562]:
xs = [[1.5,2,8,-0.7],[3.4,2.8,-7.6],[3.5,3.2,-0.1]]

pred = [0.7,0.7,-0.1]



In [591]:
for k in range(10):

#forward
 actualOuts = [a(x) for x in xs]


#backward
 loss = sum((gt-p)**2 for gt,p in zip(actualOuts,pred))
 for weights in a._weights_():
  weights.grad = 0.0

 loss.build_backward()

  #update
 for weights in a._weights_():
  weights.val += -(0.1)*weights.grad

 print(loss.val,k)

actualOuts


1.863108982772588e-09 0
1.2264082450947062e-09 1
8.073065361978439e-10 2
5.314314280869181e-10 3
3.4983265614443603e-10 4
2.3029101169921707e-10 5
1.5159905338228078e-10 6
9.979718997966162e-11 7
6.569646776748592e-11 8
4.3248122175687353e-11 9


[myNode(val=0.6999954951433345, grad=-9.009713330954128e-06),
 myNode(val=0.6999955511093716, grad=-8.897781256678172e-06),
 myNode(val=-0.0999982218659285, grad=3.5562681430234555e-06)]

In [440]:
actualOuts

[myNode(val=-0.9846934771830328, grad=0.03061304563393441),
 myNode(val=0.9499267234290047, grad=-0.1001465531419905),
 myNode(val=0.9432592697771823, grad=-0.3134814604456355)]